# Closed trade history — Bybit (last N days, M5)

Fetches **closed P&L** for `SYMBOLS`, saves `./data/<SYMBOL>/trades/closed_pnl_history.csv`, and draws **one chart per symbol**: **M5 close** over the last **5 days** (default), with entry / exit markers and dashed connectors.

**Params:** `BYBIT_TRADE_LOOKBACK_DAYS` (default `5`), `BYBIT_TRADE_CHART_INTERVAL` (default `5` = M5). Bybit closed-PnL requests use at most a **7-day** `startTime`–`endTime` range per call; longer lookbacks are stepped in slices. Platform max history (~2 years) is still capped via `BYBIT_CLOSED_PNL_PLATFORM_CAP_DAYS` if you raise the lookback.

Requires `.env` with API keys; use **`BYBIT_DEMO=true`** for Demo Trading (`api-demo.bybit.com`).

In [1]:
# %pip install plotly nbformat --quiet

In [2]:
# SECTION 1 — Imports, symbols, paths
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import time
import hashlib
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

try:
    from dotenv import load_dotenv
    _env = _ROOT / ".env"
    if _env.exists():
        load_dotenv(_env)
        print(f"Loaded .env from {_env}")
    else:
        print(f"No .env at {_env}")
except ImportError:
    print("python-dotenv not installed")

SYMBOLS = [ "BTCUSDT","XAUUSDT", "XAGUSDT" ]

# SYMBOLS = [
#     "BTCUSDT", "XRPUSDT", "ETHUSDT", "BNBUSDT", "ADAUSDT", "SOLUSDT", "DOGEUSDT",
#     "DOTUSDT", "AVAXUSDT", "BCHUSDT", "SHIB1000USDT", "LTCUSDT", "XLMUSDT",
# ]

CATEGORY = "linear"
TESTNET = os.getenv("BYBIT_TESTNET", "false").lower() == "true"
DEMO = os.getenv("BYBIT_DEMO", "true").lower() == "true"

LOOKBACK_DAYS = int(os.getenv("BYBIT_TRADE_LOOKBACK_DAYS", "5"))
CLOSED_PNL_CAP_DAYS = int(os.getenv("BYBIT_CLOSED_PNL_PLATFORM_CAP_DAYS", "719"))
EFFECTIVE_CLOSED_PNL_DAYS = min(LOOKBACK_DAYS, CLOSED_PNL_CAP_DAYS)

CHART_INTERVAL = os.getenv("BYBIT_TRADE_CHART_INTERVAL", "5")

_TEHRAN = "Asia/Tehran"
BASE_DIR = Path("./data")

_mode = "demo" if DEMO else ("testnet" if TESTNET else "mainnet")
if DEMO and TESTNET:
    raise ValueError("Use only one of BYBIT_DEMO or BYBIT_TESTNET")

print(f"Symbols: {SYMBOLS}")
print(f"Endpoint: {_mode}  |  M{CHART_INTERVAL} chart  |  lookback {LOOKBACK_DAYS}d")

Loaded .env from D:\bot\ema-h1trend-exchange\.env
Symbols: ['BTCUSDT', 'XAUUSDT', 'XAGUSDT']
Endpoint: demo  |  M5 chart  |  lookback 5d


In [3]:
# SECTION 2 — Bybit client
from pybit.unified_trading import HTTP

api_key = os.getenv("BYBIT_API_KEY", "")
api_secret = os.getenv("BYBIT_API_SECRET", "")

client = HTTP(
    testnet=TESTNET,
    demo=DEMO,
    api_key=api_key or None,
    api_secret=api_secret or None,
)

print(f"HTTP demo={DEMO} testnet={TESTNET}  |  API key set: {bool(api_key)}")

HTTP demo=True testnet=False  |  API key set: True


In [4]:
# SECTION 3 — Fetch helpers

SEVEN_MS = 7 * 24 * 60 * 60 * 1000
DAY_MS = 24 * 60 * 60 * 1000

_SL_TYPES = {"StopLoss", "Stop", "OcoTriggerByStopLoss"}
_TP_TYPES = {"TakeProfit", "OcoTriggerByTp"}


def _dedupe_key(row: dict) -> str:
    raw = "|".join(
        str(row.get(k, ""))
        for k in ("orderId", "symbol", "updatedTime", "closedSize", "avgExitPrice", "closedPnl")
    )
    return hashlib.sha256(raw.encode()).hexdigest()


def fetch_closed_pnl_all_history(symbol: str) -> list[dict]:
    now_ms = int(time.time() * 1000)
    boundary = now_ms - EFFECTIVE_CLOSED_PNL_DAYS * DAY_MS
    out: list[dict] = []
    seen: set[str] = set()

    window_end = now_ms
    while window_end > boundary:
        window_start = max(window_end - SEVEN_MS, boundary)
        cursor = None
        while True:
            kwargs = dict(
                category=CATEGORY, symbol=symbol, limit=100,
                startTime=str(window_start), endTime=str(window_end),
            )
            if cursor:
                kwargs["cursor"] = cursor

            resp = client.get_closed_pnl(**kwargs)
            if resp.get("retCode") != 0:
                raise RuntimeError(f"Bybit {resp.get('retCode')}: {resp.get('retMsg')}")

            result = resp["result"]
            items = result.get("list") or []
            for row in items:
                k = _dedupe_key(row)
                if k not in seen:
                    seen.add(k)
                    out.append(row)

            cursor = result.get("nextPageCursor") or ""
            if not cursor or not items:
                break
            time.sleep(0.06)

        window_end = window_start - 1
        time.sleep(0.06)

    return out


def closed_pnl_rows_to_dataframe(rows: list[dict]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    for col in ("createdTime", "updatedTime", "createdAt", "updatedAt"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    t_entry_col = "createdTime" if "createdTime" in df.columns else "createdAt"
    t_exit_col  = "updatedTime" if "updatedTime" in df.columns else "updatedAt"
    if t_entry_col not in df.columns or t_exit_col not in df.columns:
        raise ValueError(f"Unexpected columns: {list(df.columns)}")

    df["entry_time"] = pd.to_datetime(df[t_entry_col], unit="ms", utc=True)
    df["exit_time"]  = pd.to_datetime(df[t_exit_col],  unit="ms", utc=True)
    for col in ("avgEntryPrice", "avgExitPrice", "closedPnl", "closedSize"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "side" in df.columns:
        df["position"] = df["side"].map({"Buy": "Short", "Sell": "Long"}).fillna(df["side"])

    df = df.sort_values("exit_time").reset_index(drop=True)
    return df


def fetch_klines_in_range(symbol: str, interval: str, start_ms: int, end_ms: int) -> pd.DataFrame:
    all_rows: list[list] = []
    cur_end = int(end_ms)
    MAX_RETRIES = 5

    while cur_end >= start_ms:
        kwargs = dict(category=CATEGORY, symbol=symbol, interval=interval, limit=1000, end=cur_end)
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = client.get_kline(**kwargs)
                if resp.get("retCode") != 0:
                    raise RuntimeError(resp.get("retMsg"))
                rows = resp["result"].get("list", [])
                break
            except Exception:
                if attempt == MAX_RETRIES:
                    raise
                time.sleep(2 ** attempt)

        if not rows:
            break
        all_rows.extend(rows)
        oldest = int(rows[-1][0])
        if oldest <= start_ms or len(rows) < 1000:
            break
        cur_end = oldest - 1
        time.sleep(0.05)

    if not all_rows:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume"])

    df = pd.DataFrame(
        list(reversed(all_rows)),
        columns=["time", "open", "high", "low", "close", "volume", "turnover"],
    )
    df["time"] = pd.to_datetime(df["time"].astype("int64"), unit="ms", utc=True)
    df = df.set_index("time").sort_index()
    df = df[~df.index.duplicated(keep="last")]
    for col in ("open", "high", "low", "close", "volume"):
        df[col] = df[col].astype(float)

    start_dt = pd.to_datetime(start_ms, unit="ms", utc=True)
    end_dt   = pd.to_datetime(end_ms,   unit="ms", utc=True)
    return df.loc[(df.index >= start_dt) & (df.index <= end_dt),
                  ["open", "high", "low", "close", "volume"]]


def _classify_exit(row: pd.Series) -> str:
    sot = str(row.get("stopOrderType", "") or "").strip()
    if sot in _SL_TYPES:
        return "SL"
    if sot in _TP_TYPES:
        return "TP"
    try:
        return "SL" if float(row.get("closedPnl", 0)) < 0 else "TP"
    except (TypeError, ValueError):
        return "manual"


def compute_rsi(close: pd.Series, period: int = 21) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window=period).mean()
    loss = (-delta.clip(upper=0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - 100 / (1 + rs)


def plot_symbol_trades(symbol: str, ohlcv: pd.DataFrame, trades: pd.DataFrame) -> None:
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.72, 0.28],
        vertical_spacing=0.04,
        subplot_titles=("", "RSI (14)"),
    )

    # Candlestick — row 1
    if not ohlcv.empty:
        fig.add_trace(go.Candlestick(
            x=ohlcv.index,
            open=ohlcv["open"], high=ohlcv["high"],
            low=ohlcv["low"],  close=ohlcv["close"],
            name="Price",
            increasing_line_color="#26a69a",
            decreasing_line_color="#ef5350",
            hovertext=[
                f"O: {o:.4f}  H: {h:.4f}  L: {l:.4f}  C: {c:.4f}"
                for o, h, l, c in zip(ohlcv["open"], ohlcv["high"], ohlcv["low"], ohlcv["close"])
            ],
            hoverinfo="x+text",
        ), row=1, col=1)

    if not trades.empty:
        trades = trades.copy()
        trades["_exit_type"] = trades.apply(_classify_exit, axis=1)
        et = trades["entry_time"].dt.tz_convert(_TEHRAN)
        xt = trades["exit_time"].dt.tz_convert(_TEHRAN)

        # Dashed connectors
        for i in range(len(trades)):
            fig.add_trace(go.Scatter(
                x=[et.iloc[i], xt.iloc[i]],
                y=[float(trades["avgEntryPrice"].iloc[i]), float(trades["avgExitPrice"].iloc[i])],
                mode="lines",
                line=dict(color="rgba(150,150,150,0.5)", width=1, dash="dash"),
                showlegend=False, hoverinfo="skip",
            ), row=1, col=1)

        pos_col = trades["position"].tolist()  if "position"  in trades.columns else [""] * len(trades)
        ord_col = trades["orderType"].tolist() if "orderType" in trades.columns else [""] * len(trades)
        fig.add_trace(go.Scatter(
            x=et, y=trades["avgEntryPrice"],
            mode="markers", name="Entry",
            marker=dict(symbol="triangle-up", color="royalblue", size=13),
            customdata=list(zip(trades["avgEntryPrice"].round(4), pos_col, ord_col)),
            hovertemplate=(
                "<b>Entry</b><br>"
                "Time: %{x}<br>"
                "Price: %{customdata[0]}<br>"
                "Position: %{customdata[1]}<br>"
                "OrderType: %{customdata[2]}"
                "<extra></extra>"
            ),
        ), row=1, col=1)

        sl_mask = trades["_exit_type"] == "SL"
        sl_t = trades[sl_mask]
        if len(sl_t):
            fig.add_trace(go.Scatter(
                x=xt[sl_mask], y=sl_t["avgExitPrice"],
                mode="markers", name="Exit SL",
                marker=dict(symbol="x", color="firebrick", size=13,
                            line=dict(width=2.5, color="firebrick")),
                customdata=list(zip(sl_t["avgExitPrice"].round(4), sl_t["closedPnl"].round(4))),
                hovertemplate=(
                    "<b>Exit — Stop Loss</b><br>"
                    "Time: %{x}<br>"
                    "Price: %{customdata[0]}<br>"
                    "PnL: %{customdata[1]}"
                    "<extra></extra>"
                ),
            ), row=1, col=1)

        tp_mask = trades["_exit_type"] == "TP"
        tp_t = trades[tp_mask]
        if len(tp_t):
            fig.add_trace(go.Scatter(
                x=xt[tp_mask], y=tp_t["avgExitPrice"],
                mode="markers", name="Exit TP",
                marker=dict(symbol="circle", color="seagreen", size=12),
                customdata=list(zip(tp_t["avgExitPrice"].round(4), tp_t["closedPnl"].round(4))),
                hovertemplate=(
                    "<b>Exit — Take Profit</b><br>"
                    "Time: %{x}<br>"
                    "Price: %{customdata[0]}<br>"
                    "PnL: %{customdata[1]}"
                    "<extra></extra>"
                ),
            ), row=1, col=1)

    # RSI — row 2
    if not ohlcv.empty:
        rsi = compute_rsi(ohlcv["close"])
        fig.add_trace(go.Scatter(
            x=ohlcv.index, y=rsi,
            mode="lines", name="RSI(14)",
            line=dict(color="#b388ff", width=1.5),
            hovertemplate="RSI: %{y:.2f}<extra></extra>",
        ), row=2, col=1)
        # overbought / oversold bands
        fig.add_hrect(y0=70, y1=100, fillcolor="rgba(239,83,80,0.08)", line_width=0, row=2, col=1)
        fig.add_hrect(y0=0,  y1=30,  fillcolor="rgba(38,166,154,0.08)", line_width=0, row=2, col=1)
        fig.add_hline(y=70, line=dict(color="rgba(239,83,80,0.55)", width=1, dash="dot"), row=2, col=1)
        fig.add_hline(y=30, line=dict(color="rgba(38,166,154,0.55)", width=1, dash="dot"), row=2, col=1)

    fig.update_layout(
        title=f"{symbol}  |  last {LOOKBACK_DAYS}d  M{CHART_INTERVAL}  |  closed trades: {len(trades)}",
        hovermode="x unified",
        height=700,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        xaxis=dict(
            rangeslider=dict(visible=False),
            rangeselector=dict(buttons=[
                dict(count=6,  label="6h",  step="hour", stepmode="backward"),
                dict(count=1,  label="1D",  step="day",  stepmode="backward"),
                dict(count=3,  label="3D",  step="day",  stepmode="backward"),
                dict(step="all", label="All"),
            ]),
            type="date",
        ),
        xaxis2=dict(type="date"),
        yaxis=dict(title="Price"),
        yaxis2=dict(title="RSI", range=[0, 100], fixedrange=True),
        margin=dict(t=80, b=20),
    )
    display(fig)


print("Helpers ready.")

Helpers ready.


In [5]:
# SECTION 4 — Download closed P&L for every symbol and save CSV
trade_frames: dict[str, pd.DataFrame] = {}

for symbol in SYMBOLS:
    print(f"Fetching closed P&L: {symbol} ...")
    try:
        rows = fetch_closed_pnl_all_history(symbol)
        df = closed_pnl_rows_to_dataframe(rows)
        trade_frames[symbol] = df
        out_dir = BASE_DIR / symbol / "trades"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_csv = out_dir / "closed_pnl_history.csv"
        df.to_csv(out_csv, index=False)
        print(f"  -> {len(df)} rows  saved {out_csv}")
    except Exception as exc:
        print(f"  FAILED: {exc}")
        trade_frames[symbol] = pd.DataFrame()

print("Done fetching.")

Fetching closed P&L: BTCUSDT ...
  -> 32 rows  saved data\BTCUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: XAUUSDT ...
  -> 0 rows  saved data\XAUUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: XAGUSDT ...
  -> 2 rows  saved data\XAGUSDT\trades\closed_pnl_history.csv
Done fetching.


In [6]:
# SECTION 4b — Inspect raw trade fields (run once to verify timestamps & columns)
_debug_sym = "BTCUSDT"
_df = trade_frames.get(_debug_sym, pd.DataFrame())
if not _df.empty:
    show_cols = [c for c in [
        "entry_time", "exit_time", "avgEntryPrice", "avgExitPrice",
        "closedPnl", "side", "orderType", "stopOrderType", "orderId",
    ] if c in _df.columns]
    print(f"{_debug_sym}  columns: {list(_df.columns)}\n")
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(_df[show_cols].tail(5).to_string())
else:
    print(f"No trades for {_debug_sym}")

BTCUSDT  columns: ['symbol', 'orderType', 'leverage', 'updatedTime', 'side', 'orderId', 'closedPnl', 'openFee', 'closeFee', 'avgEntryPrice', 'qty', 'cumEntryValue', 'createdTime', 'orderPrice', 'closedSize', 'avgExitPrice', 'execType', 'fillCount', 'cumExitValue', 'entry_time', 'exit_time', 'position']

                         entry_time                        exit_time  avgEntryPrice  avgExitPrice  closedPnl  side orderType                               orderId
27 2026-05-17 00:44:17.956000+00:00 2026-05-17 00:58:45.326000+00:00        78097.0       77870.2  29.629950   Buy    Market  8e97c1e8-512e-444e-bc2f-979f9fd4b08f
28 2026-05-17 01:36:27.276000+00:00 2026-05-17 03:41:07.056000+00:00        77865.0       78025.6 -19.936924   Buy    Market  f3b2e59f-a28a-4eab-a1e3-ab59d1550e20
29 2026-05-17 06:00:05.862000+00:00 2026-05-17 06:02:17.156000+00:00        78103.7       78124.6 -13.910622   Buy    Market  e4228c41-709f-4b46-946a-b732338f894a
30 2026-05-17 06:30:03.967000+00:00 2026-05

In [7]:
# SECTION 5 — Last LOOKBACK_DAYS on M5: price + trades (one figure per symbol)
_now_ms = int(time.time() * 1000)
chart_t0 = _now_ms - LOOKBACK_DAYS * DAY_MS
chart_t1 = _now_ms
_cutoff = pd.to_datetime(chart_t0, unit="ms", utc=True)

for symbol in SYMBOLS:
    trades = trade_frames.get(symbol, pd.DataFrame())
    if not trades.empty:
        trades = trades.loc[trades["exit_time"] >= _cutoff].copy()

    print(
        f"OHLCV {symbol}  {pd.to_datetime(chart_t0, unit='ms', utc=True)} .. "
        f"{pd.to_datetime(chart_t1, unit='ms', utc=True)} UTC"
    )
    ohlcv = fetch_klines_in_range(symbol, CHART_INTERVAL, chart_t0, chart_t1)
    if not ohlcv.empty:
        ohlcv.index = ohlcv.index.tz_convert(_TEHRAN)

    plot_symbol_trades(symbol, ohlcv, trades)

print("All charts rendered.")

OHLCV BTCUSDT  2026-05-12 20:27:12.384000+00:00 .. 2026-05-17 20:27:12.384000+00:00 UTC


OHLCV XAUUSDT  2026-05-12 20:27:12.384000+00:00 .. 2026-05-17 20:27:12.384000+00:00 UTC


OHLCV XAGUSDT  2026-05-12 20:27:12.384000+00:00 .. 2026-05-17 20:27:12.384000+00:00 UTC


All charts rendered.


## Account summary — total P&L and wallet balance

Closed P&L is summed from all trades fetched in **Section 4** (up to `EFFECTIVE_CLOSED_PNL_DAYS`). Wallet balance is read live from Bybit.

In [8]:
# SECTION 6 — Total closed P&L and wallet balance

QUOTE_COIN = "USDT"
ACCOUNT_TYPE = "UNIFIED"


def _parse_wallet_usdt(resp: dict) -> dict:
    if resp.get("retCode") != 0:
        raise RuntimeError(f"Bybit {resp.get('retCode')}: {resp.get('retMsg')}")
    accounts = (resp.get("result") or {}).get("list") or []
    if not accounts:
        raise RuntimeError("Empty wallet balance response")
    coins = accounts[0].get("coin") or []
    usdt = next((c for c in coins if c.get("coin") == QUOTE_COIN), None)
    if usdt is None:
        return {
            "equity": 0.0,
            "wallet_balance": 0.0,
            "available": 0.0,
            "used_margin": 0.0,
        }
    wallet_bal = float(usdt.get("walletBalance", 0) or 0)
    position_im = float(usdt.get("totalPositionIM", 0) or 0)
    order_im = float(usdt.get("totalOrderIM", 0) or 0)
    available = max(wallet_bal - position_im - order_im, 0.0)
    return {
        "equity": float(usdt.get("equity", 0) or 0),
        "wallet_balance": wallet_bal,
        "available": available,
        "used_margin": position_im + order_im,
    }


def _pnl_summary_frames(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for sym, df in frames.items():
        if df is None or df.empty or "closedPnl" not in df.columns:
            rows.append({"symbol": sym, "trades": 0, "wins": 0, "losses": 0, "closed_pnl": 0.0})
            continue
        pnls = pd.to_numeric(df["closedPnl"], errors="coerce").fillna(0.0)
        rows.append({
            "symbol": sym,
            "trades": len(pnls),
            "wins": int((pnls > 0).sum()),
            "losses": int((pnls < 0).sum()),
            "closed_pnl": round(float(pnls.sum()), 4),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    total = pd.DataFrame([{
        "symbol": "ALL",
        "trades": int(out["trades"].sum()),
        "wins": int(out["wins"].sum()),
        "losses": int(out["losses"].sum()),
        "closed_pnl": round(float(out["closed_pnl"].sum()), 4),
    }])
    return pd.concat([out, total], ignore_index=True)


# ── Closed P&L (all symbols, full fetch window) ─────────────────────────────
pnl_by_symbol = _pnl_summary_frames(trade_frames)
total_row = pnl_by_symbol.loc[pnl_by_symbol["symbol"] == "ALL"].iloc[0] if not pnl_by_symbol.empty else None

print(f"Closed P&L window: last {EFFECTIVE_CLOSED_PNL_DAYS} day(s)  |  endpoint: {_mode}")
if total_row is not None:
    wr = (total_row["wins"] / total_row["trades"] * 100) if total_row["trades"] else 0.0
    print(
        f"Total: {int(total_row['trades'])} trades  |  "
        f"{int(total_row['wins'])}W / {int(total_row['losses'])}L  |  "
        f"win rate {wr:.1f}%  |  "
        f"net closed P&L: {total_row['closed_pnl']:+,.4f} {QUOTE_COIN}"
    )
pnl_show = pnl_by_symbol.copy()
pnl_show["closed_pnl"] = pnl_show["closed_pnl"].map(lambda x: f"{x:+,.4f}")
display(pnl_show)

# ── Wallet balance (live) ───────────────────────────────────────────────────
wallet_resp = client.get_wallet_balance(accountType=ACCOUNT_TYPE, coin=QUOTE_COIN)
wallet = _parse_wallet_usdt(wallet_resp)

wallet_df = pd.DataFrame([{
    "equity": f"{wallet['equity']:,.2f}",
    "wallet_balance": f"{wallet['wallet_balance']:,.2f}",
    "available": f"{wallet['available']:,.2f}",
    "used_margin": f"{wallet['used_margin']:,.2f}",
}])
print(f"\nWallet ({ACCOUNT_TYPE}, {QUOTE_COIN}):")
display(wallet_df)

Closed P&L window: last 5 day(s)  |  endpoint: demo
Total: 34 trades  |  13W / 21L  |  win rate 38.2%  |  net closed P&L: -212.8902 USDT


,symbol,trades,wins,losses,closed_pnl
0,BTCUSDT,32,12,20,-217.4065
1,XAUUSDT,0,0,0,+0.0000
2,XAGUSDT,2,1,1,+4.5163
3,ALL,34,13,21,-212.8902



Wallet (UNIFIED, USDT):


,equity,wallet_balance,available,used_margin
0,"56,174.66","56,180.86","54,785.65","1,395.21"
